In [ ]:
import numpy as np
from data import tracks_data
from data import measurement_data
from data import vars
from target_tracking.target_track import Track, TrackManager
from target_tracking.kalman_filter import KalmanFilter
import time

In [ ]:
def measurement_noise(R):
    return np.random.multivariate_normal(mean=[0, 0], cov=vars.R).reshape(2, 1)

In [ ]:
P0 = np.diag([vars.R[0,0], vars.R[1.0,1.0], 100.0, 100.0])

In [ ]:
true_state = []
truth_data = [ 
                {
        "id": 1,
        "x": np.array([[80.0],
                       [111.0],
                       [  3.0],
                       [ 0.0]], dtype=float),
        "P": P0
    },
    {
        "id": 2,
        "x": np.array([[25.0],
                       [111.0],
                       [  1.0],
                       [ -1.0]], dtype=float),
        "P": P0
    },
    {
        "id": 3,
        "x": np.array([[10.0],
                       [80.0],
                       [  1.0],
                       [ -1.0]], dtype=float),
        "P": P0
    },
    {
        "id": 4,
        "x": np.array([[20.0],
                       [140.0],
                       [  1.0],
                       [ 1.0]], dtype=float),
        "P": P0
    },
    {
        "id": 5,
        "x": np.array([[80.0],
                       [140.0],
                       [  -1.0],
                       [ 1.0]], dtype=float),
        "P": P0
    }
]
{
        "id": 1,
        "x": np.array([[80.0],
                       [111.0],
                       [  3.0],
                       [ 0.0]], dtype=float),
        "P": P0
},
n_steps = 100
track_truths = []
measure_data = []
for state in truth_data:
    x_k = state['x'].copy()
    truth_states = {"id": state['id'], "x_states": [x_k.copy()], "P": P0}
    track_measurements = {"id": state['id']}
    for _ in range(n_steps):
        x_k = vars.F @ x_k
        truth_states['x_states'].append(x_k.copy())
        if not 'measurements' in track_measurements.keys():
            track_measurements['measurements'] = [vars.H @ x_k.copy() + measurement_noise(vars.R)]
        else:
            track_measurements['measurements'].append(vars.H @ x_k.copy() + measurement_noise(vars.R))
    track_truths.append(truth_states)
    measure_data.append(track_measurements)

In [ ]:
scans = []
for i, (v, w, x, y, z) in enumerate(zip(*[md['measurements'] for md in measure_data])):
    if i in [10, 11, 12, 20, 30, 40, 51, 52]: # should drop track 1 at k+11 and find a new track
        scans.append({f"scan_k{i}" : [w, x, y, z]})
    else:    
        scans.append({f"scan_k{i}" : [v, w, x, y, z]})

In [ ]:
trax = [
    Track(
        track['id'],
        KalmanFilter(F=vars.F, H=vars.H, R=vars.R, Q=vars.Q, x_hat_km1_km1=track["x"], P_km1_km1=track["P"])
    )
    for track in truth_data
]

In [ ]:
def angle_between(v1, v2):
    v1_u = v1 / np.linalg.norm(v1)
    v2_u = v2 / np.linalg.norm(v2)
    dot_product = np.dot(v1_u, v2_u)
    angle_rad = np.arccos(np.clip(dot_product, -1.0, 1.0))
    
    return np.degrees(angle_rad)


trax = [
    Track(
        track['id'],
        KalmanFilter(F=vars.F, H=vars.H, R=vars.R, Q=vars.Q, x_hat_km1_km1=track["x"], P_km1_km1=track["P"])
    )
    for track in truth_data
] # Generate initital track, measurement model, and motion model with associated process / measurement noise covariance.

tracker = TrackManager(trax, 5.991)

measurement_train = [measurement_data.measurements,
                     measurement_data.measurements_kp1,
                     measurement_data.measurements_kp2,
                     measurement_data.measurements_kp3]

for ms in measurement_train:
    tracker.predict_all()
    assignments, unassigned_tracks, unassigned_measurements, _ = tracker.gnn_associate(ms)

    assignment_map = {a['track_id']: a['measurement'] for a in assignments} # key = track id , val = measurement from ass.
    unassigned_track_ids = {t['track_id'] for t in unassigned_tracks} # track id of unassigned track

    tracks_to_delete = []

    for trk in tracker.tracks: # for ech track
        if type(trk.kf.x_hat_k_k) is np.ndarray and type(trk.kf.x_hat_k_km1) is np.ndarray:
            print(f"Track ID: {trk.track_id} x_hat_prior, x_hat_predicted angle between {angle_between(trk.kf.x_hat_km1_km1.flatten(), trk.kf.x_hat_k_km1.flatten())}")
            print(f"Track ID {trk.track_id} Trace: P_prior: {np.trace(trk.kf.P_km1_km1)} -> P_predicted: {np.trace(trk.kf.P_k_km1)}")
        time.sleep(1.5)
        if trk.track_id in assignment_map: 
            z_k = assignment_map[trk.track_id] # Get measurement from ass map

            trk.update(z_k) # use measurement / prediction to update 

            if trk.tenative: # if its a tenative track increase hit count by 1
                trk.hit_count += 1
                if trk.hit_count >= 2: # promote track if hit count >= 2
                    trk.promote_track()
                    print(f"Tenative Track ID: {trk.track_id} Promoted! Hit Count: {trk.hit_count}")
            print(f"Track ID {trk.track_id} Trace: P_predicted: {np.trace(trk.kf.P_k_km1)} -> P_updated: {np.trace(trk.kf.P_k_k)}")
            print(f"Track ID: {trk.track_id} x_hat_predicted, x_hat_updated angle between {angle_between(trk.kf.x_hat_km1_km1.flatten(), trk.kf.x_hat_k_km1.flatten())}")
            print(f"Track {trk.track_id} Detected!")

        elif trk.track_id in unassigned_track_ids: # if this track id is unassigned then mark as a miss and propagate
                                                    # predication from k|k-1 -> k|k
            trk.miss()
            trk.propagate()
            print(f"Track {trk.track_id} Missed Detection!")

        if trk.tenative and trk.missed_count >= 2: # delete a tenative track with missed count >= 2
            print(f"Tenative Track ID: {trk.track_id} Deleted, Missed Count: {trk.missed_count} >= 2")
            tracks_to_delete.append(trk)

        elif (not trk.tenative) and trk.missed_count >= 3:
            # delete a confirmed track with miss count > = 3
            print(f"Confirmed Track ID: {trk.track_id} Deleted, Missed Count: {trk.missed_count} >= 3")
            tracks_to_delete.append(trk)

    for trk in tracks_to_delete: # accutally delete the tracks from the list.
        tracker.delete_track(trk)

    for meas in unassigned_measurements:
        # For each unassigned measurement create a tenative track
        ten_track_id = tracker.tenative_track(meas['measurement'], tracker.get_new_track_id())
        print(f"Tenative Track ID: {ten_track_id} Created!")



In [ ]:
truth_time = None
turth_state = None
truth_position = None
truth_velocity = None

